# Notebook 28 — Lithium Plating External Diagnostic Closure Contract

## Goal

This notebook defines the design contract for closing the external diagnostic fingerprint of lithium plating.

It does not execute new PyBaMM simulations.

The goal is to specify:

- claim ladder（声明层级）
- matched-control condition registry（匹配对照条件表）
- diagnostic layers（诊断层）
- output schemas（输出表结构）
- decision rules（判据）
- process-isolated execution plan（进程隔离执行计划）

## Scientific context

Previous plating notebooks established that lithium plating variables are observable inside the model-based audit workflow.

The current open question is different:

> Can lithium plating produce stable, measurement-accessible diagnostic fingerprints that are suitable as fitting-target candidates?

The distinction is critical:

- model-internal plating-state observability（模型内部析锂状态可观测性）is already supported
- external diagnostic fingerprint closure（外部诊断指纹闭合）remains deferred

Notebook 28 therefore moves from model-internal observability toward externally accessible diagnostic layers.

## Protocol / study design

This notebook defines a contract for future process-isolated execution.

The planned diagnostic layers are:

1. C/25 central-window V(Q)（C/25 中央窗口电压曲线）
2. GITT-like finite-rest voltage（GITT-like 有限静置电压）
3. ICA = Incremental Capacity Analysis（增量容量分析）
4. DVA = Differential Voltage Analysis（微分电压分析）

The diagnostic design uses matched controls:

- plating-enabled condition
- no-plating control condition
- same temperature
- same stress charge rate
- same diagnostic protocol
- same post-stress state handling

## Workflow

1. Audit repository state and output paths.
2. Define claim ladder from model-internal observability to fitting-target candidate.
3. Define matched-control condition registry.
4. Define diagnostic-layer schema.
5. Define decision-rule registry.
6. Define process-isolated execution plan.
7. Save contract tables to `data/` and `docs/tables/`.
8. Close with interpretation boundaries and next action.

## Expected outputs

This notebook should generate contract-level tables only:

- `data/plating_external_diagnostic_condition_registry_v0_1.csv`
- `data/plating_external_diagnostic_schema_v0_1.csv`
- `data/plating_external_diagnostic_decision_rules_v0_1.csv`
- `docs/tables/plating_external_diagnostic_condition_registry_v0_1.csv`
- `docs/tables/plating_external_diagnostic_schema_v0_1.csv`
- `docs/tables/plating_external_diagnostic_decision_rules_v0_1.csv`
- `data/plating_external_claim_ladder_v0_1.csv`
- `docs/tables/plating_external_claim_ladder_v0_1.csv`
- `data/plating_external_diagnostic_feature_plan_v0_1.csv`
- `data/plating_external_process_isolated_execution_plan_v0_1.csv`
- `data/plating_external_contract_inventory_v0_1.csv`
- `docs/tables/plating_external_claim_ladder_v0_1.csv`
- `docs/tables/plating_external_diagnostic_feature_plan_v0_1.csv`
- `docs/tables/plating_external_process_isolated_execution_plan_v0_1.csv`
- `docs/tables/plating_external_contract_inventory_v0_1.csv`

No simulation output is expected from this notebook.

## Interpretation boundary

This notebook does not establish a new lithium-plating diagnostic result.

It only freezes the design contract for future execution.

Any future claim of plating-associated external diagnostic fingerprints must pass matched-control, central-window, endpoint-amplification, smoothing-sensitivity, and protocol-boundary checks.

Kernel death, memory pressure, or solver failure must be classified as execution boundary, not as scientific negative evidence.

In [1]:
# Cell 2 — Environment and repository audit（环境与仓库状态审计）

from pathlib import Path
import sys
import subprocess
import platform
import pandas as pd
import numpy as np

cwd = Path.cwd()
expected_repo_name = "pybamm-aging-bridge"

# Robust repository-root detection
if cwd.name == expected_repo_name:
    repo = cwd
elif cwd.name == "notebooks" and cwd.parent.name == expected_repo_name:
    repo = cwd.parent
else:
    repo = cwd

print("[ENV] Python executable:")
print(sys.executable)

print("\n[ENV] Python version:")
print(sys.version)

print("\n[ENV] Platform:")
print(platform.platform())

print("\n[PATH] Notebook current working directory:")
print(cwd)

print("\n[PATH] Resolved repository root:")
print(repo)

if repo.name != expected_repo_name:
    print(f"[WARNING] Resolved repo name is '{repo.name}', expected '{expected_repo_name}'.")
else:
    print(f"[OK] Repository root resolved correctly: {repo}")

print("\n[GIT] status:")
git_status = subprocess.run(
    ["git", "status", "--short"],
    cwd=repo,
    capture_output=True,
    text=True,
)
print(git_status.stdout if git_status.stdout.strip() else "[OK] working tree clean")

print("\n[GIT] recent commits:")
git_log = subprocess.run(
    ["git", "log", "--oneline", "--max-count=8"],
    cwd=repo,
    capture_output=True,
    text=True,
)
print(git_log.stdout)

print("\n[GIT] remote:")
git_remote = subprocess.run(
    ["git", "remote", "-v"],
    cwd=repo,
    capture_output=True,
    text=True,
)
print(git_remote.stdout if git_remote.stdout.strip() else "[WARNING] no git remote found")

# Output folders
data_dir = repo / "data"
docs_dir = repo / "docs"
docs_table_dir = docs_dir / "tables"

for path in [data_dir, docs_dir, docs_table_dir]:
    path.mkdir(parents=True, exist_ok=True)
    print(f"[OK] ensured directory: {path}")

# Package audit
print("\n[PKG] Core package versions:")
for pkg in ["pybamm", "pandas", "numpy"]:
    try:
        mod = __import__(pkg)
        print(f"[OK] {pkg}: {mod.__version__}")
    except Exception as exc:
        print(f"[WARNING] {pkg}: not available ({exc})")

# Contract boundary
print("\n[BOUNDARY] Execution boundary:")
print("[OK] This notebook defines the diagnostic-closure contract only.")
print("[OK] No PyBaMM simulation is executed in this notebook.")
print("[OK] Future heavy diagnostics must use process-isolated execution.")

# Sanity assertions
assert repo.name == expected_repo_name, "Repository root was not resolved correctly."
assert data_dir.parent == repo, "data_dir is not under repository root."
assert docs_table_dir.parent == docs_dir, "docs_table_dir is not under docs/."

print("\n[OK] Environment and repository audit completed.")

[ENV] Python executable:
/Users/louislu/projects/pybamm-aging-bridge/.venv/bin/python

[ENV] Python version:
3.11.15 (main, Mar  3 2026, 00:52:57) [Clang 21.0.0 (clang-2100.0.123.102)]

[ENV] Platform:
macOS-26.3.1-arm64-arm-64bit

[PATH] Notebook current working directory:
/Users/louislu/projects/pybamm-aging-bridge/notebooks

[PATH] Resolved repository root:
/Users/louislu/projects/pybamm-aging-bridge
[OK] Repository root resolved correctly: /Users/louislu/projects/pybamm-aging-bridge

[GIT] status:
?? notebooks/28_lithium_plating_external_diagnostic_closure_contract.ipynb


[GIT] recent commits:
1f95794 Update README.md
447af1a docs: refactor README to v0.1 release entry, add MIT License
20944b4 chore: pin requirements, refresh v0.1 heatmap and synthesis notebook
39910e8 docs: prepare repository overview for release
d049cb5 docs: add cross-mechanism fingerprint synthesis notebook
72b81c3 docs: add cross-mechanism status table
5b85a24 docs: close plating HPPC process-isolated noteboo

## 1. Claim ladder（声明层级）

This block defines the claim ladder for lithium plating external diagnostic closure.

The purpose is to prevent overclaiming.

A plating-related signal can appear at different evidence levels:

| Level | Meaning |
|---|---|
| Level 0 | Model-internal plating-state observability（模型内部析锂状态可观测） |
| Level 1 | Matched external signal（匹配对照外部信号） |
| Level 2 | Candidate external plating fingerprint（候选外部析锂指纹） |
| Level 3 | Fitting-target candidate（拟合目标候选） |

Notebook 25, 26, and 26B already support Level 0.

Notebook 28 does not need to prove again that internal plating variables exist. Instead, it defines the criteria for moving from internal observability toward measurement-accessible diagnostic fingerprints.

Important distinction:

- Model-internal plating variables are available inside the simulation workflow.
- Measurement-accessible diagnostic features must be inferred from voltage, current, capacity, relaxation, or derivative features.
- These two evidence layers must not be merged.

A future diagnostic feature can only move upward in the claim ladder if it passes matched-control, central-window, endpoint-amplification, smoothing-sensitivity, and protocol-boundary checks.

In [2]:
# Cell 4 — Define and save claim ladder registry（定义并保存声明层级注册表）

claim_ladder_rows = [
    {
        "claim_level": "Level 0",
        "claim_name": "model-internal plating-state observability",
        "evidence_object": "model-internal plating-state variables",
        "minimum_evidence_requirement": (
            "Plating-enabled model branch shows non-zero plating-state variables "
            "with stress sensitivity and clean matched-control separation."
        ),
        "allowed_conclusion_wording": (
            "Model-internal lithium-plating state observability is supported."
        ),
        "forbidden_overclaim": (
            "Do not state that lithium plating is directly observable in external measurements."
        ),
        "current_status": "already_supported",
        "current_basis": "Notebook 25 / Notebook 26 Phase A direct plating-state audits",
    },
    {
        "claim_level": "Level 1",
        "claim_name": "matched external signal",
        "evidence_object": (
            "measurement-accessible diagnostic feature under plating-enabled branch "
            "minus matched no-plating control"
        ),
        "minimum_evidence_requirement": (
            "Same temperature, same stress charge rate, same diagnostic protocol, "
            "same post-stress state handling; matched delta is quantitatively detectable."
        ),
        "allowed_conclusion_wording": (
            "A matched external diagnostic signal is detected under the tested model and protocol conditions."
        ),
        "forbidden_overclaim": (
            "Do not call the feature a fitting target or a robust plating fingerprint yet."
        ),
        "current_status": "deferred",
        "current_basis": "To be tested in future process-isolated diagnostic execution",
    },
    {
        "claim_level": "Level 2",
        "claim_name": "candidate external plating fingerprint",
        "evidence_object": (
            "stable external diagnostic feature after central-window, endpoint, "
            "smoothing, and protocol-boundary checks"
        ),
        "minimum_evidence_requirement": (
            "Matched signal is not endpoint-only, direction is stable, smoothing sensitivity is acceptable, "
            "and signal magnitude exceeds preprocessing / protocol sensitivity."
        ),
        "allowed_conclusion_wording": (
            "A candidate external lithium-plating diagnostic fingerprint is supported under the tested conditions."
        ),
        "forbidden_overclaim": (
            "Do not claim universal transferability or mechanism uniqueness without contrast branches."
        ),
        "current_status": "deferred",
        "current_basis": "Requires diagnostic-layer audit for C/25 V(Q), GITT-like voltage, ICA, and DVA",
    },
    {
        "claim_level": "Level 3",
        "claim_name": "fitting-target candidate",
        "evidence_object": (
            "external diagnostic feature qualified for model parameterization or calibration"
        ),
        "minimum_evidence_requirement": (
            "Feature is stable, measurement-accessible, linked to plating state, separated from matched controls, "
            "not dominated by endpoint or smoothing artifacts, and not easily reproduced by simpler mechanisms."
        ),
        "allowed_conclusion_wording": (
            "The feature is a candidate fitting target for lithium-plating-sensitive model calibration."
        ),
        "forbidden_overclaim": (
            "Do not treat a single observable as uniquely identifying plating without multi-evidence support."
        ),
        "current_status": "not_targeted_in_this_contract",
        "current_basis": (
            "Notebook 28 contract targets Level 1–2 closure first; Level 3 requires later validation."
        ),
    },
]

claim_ladder_df = pd.DataFrame(claim_ladder_rows)

claim_ladder_data_path = data_dir / "plating_external_claim_ladder_v0_1.csv"
claim_ladder_docs_path = docs_table_dir / "plating_external_claim_ladder_v0_1.csv"

claim_ladder_df.to_csv(claim_ladder_data_path, index=False)
claim_ladder_df.to_csv(claim_ladder_docs_path, index=False)

print(f"[OK] saved claim ladder registry: {claim_ladder_data_path}")
print(f"[OK] saved docs claim ladder registry: {claim_ladder_docs_path}")

print("\n[OK] Claim ladder registry:")
display(claim_ladder_df)

# Audit checks
assert claim_ladder_df["claim_level"].is_unique, "Each claim level must be unique."
assert "already_supported" in set(claim_ladder_df["current_status"]), (
    "Claim ladder must preserve existing model-internal plating-state support."
)
assert (
    claim_ladder_df.loc[
        claim_ladder_df["claim_level"] == "Level 0",
        "forbidden_overclaim"
    ]
    .iloc[0]
    .lower()
    .find("external measurements") >= 0
), "Level 0 must explicitly forbid direct experimental-observability overclaim."

assert (
    claim_ladder_df.loc[
        claim_ladder_df["claim_level"] == "Level 3",
        "current_status"
    ]
    .iloc[0]
    == "not_targeted_in_this_contract"
), "Level 3 should not be targeted in this contract notebook."

print("\n[BOUNDARY CHECKS]")
print("[OK] Level 0 remains model-internal only.")
print("[OK] Level 1–2 are deferred to future process-isolated diagnostic execution.")
print("[OK] Level 3 fitting-target qualification is not claimed in this notebook.")
print("[OK] Claim ladder prevents merging model-internal observability with external diagnostic closure.")

[OK] saved claim ladder registry: /Users/louislu/projects/pybamm-aging-bridge/data/plating_external_claim_ladder_v0_1.csv
[OK] saved docs claim ladder registry: /Users/louislu/projects/pybamm-aging-bridge/docs/tables/plating_external_claim_ladder_v0_1.csv

[OK] Claim ladder registry:


,claim_level,claim_name,evidence_object,minimum_evidence_requirement,allowed_conclusion_wording,forbidden_overclaim,current_status,current_basis
0,Level 0,model-internal plating-state observability,model-internal plating-state variables,Plating-enabled model branch shows non-zero pl...,Model-internal lithium-plating state observabi...,Do not state that lithium plating is directly ...,already_supported,Notebook 25 / Notebook 26 Phase A direct plati...
1,Level 1,matched external signal,measurement-accessible diagnostic feature unde...,"Same temperature, same stress charge rate, sam...",A matched external diagnostic signal is detect...,Do not call the feature a fitting target or a ...,deferred,To be tested in future process-isolated diagno...
2,Level 2,candidate external plating fingerprint,stable external diagnostic feature after centr...,"Matched signal is not endpoint-only, direction...",A candidate external lithium-plating diagnosti...,Do not claim universal transferability or mech...,deferred,"Requires diagnostic-layer audit for C/25 V(Q),..."
3,Level 3,fitting-target candidate,external diagnostic feature qualified for mode...,"Feature is stable, measurement-accessible, lin...",The feature is a candidate fitting target for ...,Do not treat a single observable as uniquely i...,not_targeted_in_this_contract,Notebook 28 contract targets Level 1–2 closure...



[BOUNDARY CHECKS]
[OK] Level 0 remains model-internal only.
[OK] Level 1–2 are deferred to future process-isolated diagnostic execution.
[OK] Level 3 fitting-target qualification is not claimed in this notebook.
[OK] Claim ladder prevents merging model-internal observability with external diagnostic closure.


## 2. Matched-control condition registry（匹配对照条件注册表）

This block defines the condition set for future lithium-plating external diagnostic closure.

The registry is based on the stress conditions already used in previous plating audits:

- 25 °C / 1C
- 10 °C / 1C
- 10 °C / 2C

Each plating-enabled condition must have a matched no-plating control.

Matched-control requirements:

- same temperature
- same stress charge rate
- same stress protocol
- same post-stress diagnostic protocol
- same post-stress state handling

This registry does not run simulations. It freezes the planned comparison structure for future process-isolated execution.

Important boundary:

The condition registry preserves direct plating-variable evidence as expected internal support, but the external diagnostic layers remain deferred until diagnostic scripts generate measurement-accessible features.

In [3]:
# Cell 6 — Build and save matched-control condition registry（构建并保存匹配对照条件注册表）

condition_rows = [
    {
        "condition_id": "no_plating_25C_1C",
        "plating_enabled": False,
        "temperature_C": 25,
        "stress_charge_rate_C": 1.0,
        "control_condition_id": "",
        "matched_pair_id": "pair_25C_1C",
        "stress_protocol_id": "stress_charge_to_4p2V_CV_to_C20_rest_discharge_rest",
        "diagnostic_protocols_planned": "C25_VQ; GITT_like_finite_rest; ICA; DVA",
        "expected_internal_plating_signal": "none",
        "expected_external_signal_strength": "control_reference",
        "execution_status": "planned",
        "notes": "Matched no-plating control for 25C / 1C stress case.",
    },
    {
        "condition_id": "plating_25C_1C",
        "plating_enabled": True,
        "temperature_C": 25,
        "stress_charge_rate_C": 1.0,
        "control_condition_id": "no_plating_25C_1C",
        "matched_pair_id": "pair_25C_1C",
        "stress_protocol_id": "stress_charge_to_4p2V_CV_to_C20_rest_discharge_rest",
        "diagnostic_protocols_planned": "C25_VQ; GITT_like_finite_rest; ICA; DVA",
        "expected_internal_plating_signal": "supported_weak_to_moderate",
        "expected_external_signal_strength": "unknown_deferred",
        "execution_status": "planned",
        "notes": "Prior audits showed model-internal plating signal under 25C / 1C, but external fingerprint remains open.",
    },
    {
        "condition_id": "no_plating_10C_1C",
        "plating_enabled": False,
        "temperature_C": 10,
        "stress_charge_rate_C": 1.0,
        "control_condition_id": "",
        "matched_pair_id": "pair_10C_1C",
        "stress_protocol_id": "stress_charge_to_4p2V_CV_to_C20_rest_discharge_rest",
        "diagnostic_protocols_planned": "C25_VQ; GITT_like_finite_rest; ICA; DVA",
        "expected_internal_plating_signal": "none",
        "expected_external_signal_strength": "control_reference",
        "execution_status": "planned",
        "notes": "Matched no-plating control for 10C / 1C stress case.",
    },
    {
        "condition_id": "plating_10C_1C",
        "plating_enabled": True,
        "temperature_C": 10,
        "stress_charge_rate_C": 1.0,
        "control_condition_id": "no_plating_10C_1C",
        "matched_pair_id": "pair_10C_1C",
        "stress_protocol_id": "stress_charge_to_4p2V_CV_to_C20_rest_discharge_rest",
        "diagnostic_protocols_planned": "C25_VQ; GITT_like_finite_rest; ICA; DVA",
        "expected_internal_plating_signal": "supported_moderate",
        "expected_external_signal_strength": "unknown_deferred",
        "execution_status": "planned",
        "notes": "Prior audits showed stronger model-internal plating signal than 25C / 1C.",
    },
    {
        "condition_id": "no_plating_10C_2C",
        "plating_enabled": False,
        "temperature_C": 10,
        "stress_charge_rate_C": 2.0,
        "control_condition_id": "",
        "matched_pair_id": "pair_10C_2C",
        "stress_protocol_id": "stress_charge_to_4p2V_CV_to_C20_rest_discharge_rest",
        "diagnostic_protocols_planned": "C25_VQ; GITT_like_finite_rest; ICA; DVA",
        "expected_internal_plating_signal": "none",
        "expected_external_signal_strength": "control_reference",
        "execution_status": "planned",
        "notes": "Matched no-plating control for 10C / 2C stress case.",
    },
    {
        "condition_id": "plating_10C_2C",
        "plating_enabled": True,
        "temperature_C": 10,
        "stress_charge_rate_C": 2.0,
        "control_condition_id": "no_plating_10C_2C",
        "matched_pair_id": "pair_10C_2C",
        "stress_protocol_id": "stress_charge_to_4p2V_CV_to_C20_rest_discharge_rest",
        "diagnostic_protocols_planned": "C25_VQ; GITT_like_finite_rest; ICA; DVA",
        "expected_internal_plating_signal": "supported_strongest_in_current_matrix",
        "expected_external_signal_strength": "unknown_deferred",
        "execution_status": "planned",
        "notes": "Highest-stress condition in current plating matrix; expected to maximize diagnostic separability if external signal exists.",
    },
]

condition_registry_df = pd.DataFrame(condition_rows)

condition_registry_data_path = data_dir / "plating_external_diagnostic_condition_registry_v0_1.csv"
condition_registry_docs_path = docs_table_dir / "plating_external_diagnostic_condition_registry_v0_1.csv"

condition_registry_df.to_csv(condition_registry_data_path, index=False)
condition_registry_df.to_csv(condition_registry_docs_path, index=False)

print(f"[OK] saved condition registry: {condition_registry_data_path}")
print(f"[OK] saved docs condition registry: {condition_registry_docs_path}")

print("\n[OK] Condition registry:")
display(condition_registry_df)

# Compact matched-pair audit table
matched_pair_audit_df = (
    condition_registry_df
    .groupby("matched_pair_id")
    .agg(
        n_conditions=("condition_id", "count"),
        n_plating_enabled=("plating_enabled", "sum"),
        temperatures_C=("temperature_C", lambda x: sorted(set(x))),
        stress_rates_C=("stress_charge_rate_C", lambda x: sorted(set(x))),
    )
    .reset_index()
)

print("\n[OK] Matched-pair audit:")
display(matched_pair_audit_df)

# Audit checks
assert condition_registry_df["condition_id"].is_unique, "condition_id values must be unique."

assert set(condition_registry_df["plating_enabled"]) == {True, False}, (
    "Registry must include both plating-enabled and no-plating control conditions."
)

for pair_id, grp in condition_registry_df.groupby("matched_pair_id"):
    assert len(grp) == 2, f"{pair_id} must contain exactly two conditions."
    assert grp["plating_enabled"].sum() == 1, f"{pair_id} must contain exactly one plating-enabled condition."
    assert grp["temperature_C"].nunique() == 1, f"{pair_id} must use the same temperature."
    assert grp["stress_charge_rate_C"].nunique() == 1, f"{pair_id} must use the same stress charge rate."

plating_rows = condition_registry_df[condition_registry_df["plating_enabled"]]
assert plating_rows["control_condition_id"].ne("").all(), (
    "Every plating-enabled condition must reference a matched no-plating control."
)

planned_layers = set()
for entry in condition_registry_df["diagnostic_protocols_planned"]:
    planned_layers.update([x.strip() for x in entry.split(";")])

expected_layers = {"C25_VQ", "GITT_like_finite_rest", "ICA", "DVA"}

assert expected_layers.issubset(planned_layers), (
    f"Missing planned diagnostic layers: {expected_layers - planned_layers}"
)

print("\n[BOUNDARY CHECKS]")
print("[OK] Each matched pair contains one plating-enabled branch and one no-plating control.")
print("[OK] Temperature and stress charge rate are matched within every pair.")
print("[OK] Every plating-enabled branch references a matched control.")
print("[OK] All four planned diagnostic layers are registered.")
print("[OK] Registry is a design contract and does not execute simulations.")

[OK] saved condition registry: /Users/louislu/projects/pybamm-aging-bridge/data/plating_external_diagnostic_condition_registry_v0_1.csv
[OK] saved docs condition registry: /Users/louislu/projects/pybamm-aging-bridge/docs/tables/plating_external_diagnostic_condition_registry_v0_1.csv

[OK] Condition registry:


,condition_id,plating_enabled,temperature_C,stress_charge_rate_C,control_condition_id,matched_pair_id,stress_protocol_id,diagnostic_protocols_planned,expected_internal_plating_signal,expected_external_signal_strength,execution_status,notes
0,no_plating_25C_1C,False,25,1.0,,pair_25C_1C,stress_charge_to_4p2V_CV_to_C20_rest_discharge...,C25_VQ; GITT_like_finite_rest; ICA; DVA,none,control_reference,planned,Matched no-plating control for 25C / 1C stress...
1,plating_25C_1C,True,25,1.0,no_plating_25C_1C,pair_25C_1C,stress_charge_to_4p2V_CV_to_C20_rest_discharge...,C25_VQ; GITT_like_finite_rest; ICA; DVA,supported_weak_to_moderate,unknown_deferred,planned,Prior audits showed model-internal plating sig...
2,no_plating_10C_1C,False,10,1.0,,pair_10C_1C,stress_charge_to_4p2V_CV_to_C20_rest_discharge...,C25_VQ; GITT_like_finite_rest; ICA; DVA,none,control_reference,planned,Matched no-plating control for 10C / 1C stress...
3,plating_10C_1C,True,10,1.0,no_plating_10C_1C,pair_10C_1C,stress_charge_to_4p2V_CV_to_C20_rest_discharge...,C25_VQ; GITT_like_finite_rest; ICA; DVA,supported_moderate,unknown_deferred,planned,Prior audits showed stronger model-internal pl...
4,no_plating_10C_2C,False,10,2.0,,pair_10C_2C,stress_charge_to_4p2V_CV_to_C20_rest_discharge...,C25_VQ; GITT_like_finite_rest; ICA; DVA,none,control_reference,planned,Matched no-plating control for 10C / 2C stress...
5,plating_10C_2C,True,10,2.0,no_plating_10C_2C,pair_10C_2C,stress_charge_to_4p2V_CV_to_C20_rest_discharge...,C25_VQ; GITT_like_finite_rest; ICA; DVA,supported_strongest_in_current_matrix,unknown_deferred,planned,Highest-stress condition in current plating ma...



[OK] Matched-pair audit:


,matched_pair_id,n_conditions,n_plating_enabled,temperatures_C,stress_rates_C
0,pair_10C_1C,2,1,[10],[1.0]
1,pair_10C_2C,2,1,[10],[2.0]
2,pair_25C_1C,2,1,[25],[1.0]



[BOUNDARY CHECKS]
[OK] Each matched pair contains one plating-enabled branch and one no-plating control.
[OK] Temperature and stress charge rate are matched within every pair.
[OK] Every plating-enabled branch references a matched control.
[OK] All four planned diagnostic layers are registered.
[OK] Registry is a design contract and does not execute simulations.


## 3. Diagnostic-layer schema（诊断层输出结构）

This block defines the required output schema for future process-isolated lithium-plating external diagnostic scripts.

The schema covers four planned diagnostic layers:

1. C/25 central-window V(Q)（C/25 中央窗口电压曲线）
2. GITT-like finite-rest voltage（GITT-like 有限静置电压）
3. ICA = Incremental Capacity Analysis（增量容量分析）
4. DVA = Differential Voltage Analysis（微分电压分析）

The purpose is to make future outputs comparable across matched controls.

The future descriptor table must distinguish:

- raw feature value（原始特征值）
- matched delta（相对匹配对照的差值）
- window definition（窗口定义）
- preprocessing setting（预处理设置）
- boundary flags（边界标记）
- claim status（声明状态）

Important boundary:

This schema does not imply that any feature is already supported. It only defines how future results must be reported.

In [4]:
# Cell 8 — Define and save diagnostic descriptor schema（定义并保存诊断描述符输出结构）

schema_rows = [
    {
        "column_name": "condition_id",
        "dtype": "string",
        "required": True,
        "description": "Condition identifier from the matched-control registry.",
        "example": "plating_10C_2C",
    },
    {
        "column_name": "control_condition_id",
        "dtype": "string",
        "required": True,
        "description": "Matched no-plating control condition. Empty only for control rows.",
        "example": "no_plating_10C_2C",
    },
    {
        "column_name": "matched_pair_id",
        "dtype": "string",
        "required": True,
        "description": "Matched pair identifier shared by plating-enabled and no-plating control branches.",
        "example": "pair_10C_2C",
    },
    {
        "column_name": "diagnostic_layer",
        "dtype": "category",
        "required": True,
        "description": "Diagnostic layer being evaluated.",
        "example": "C25_VQ",
    },
    {
        "column_name": "feature_name",
        "dtype": "string",
        "required": True,
        "description": "Name of extracted diagnostic feature.",
        "example": "central_mean_delta_U",
    },
    {
        "column_name": "feature_value",
        "dtype": "float",
        "required": True,
        "description": "Feature value for the current condition.",
        "example": "-2.35",
    },
    {
        "column_name": "feature_unit",
        "dtype": "string",
        "required": True,
        "description": "Unit of the feature value.",
        "example": "mV",
    },
    {
        "column_name": "matched_delta",
        "dtype": "float",
        "required": True,
        "description": "Plating-enabled feature minus matched no-plating control feature.",
        "example": "-1.12",
    },
    {
        "column_name": "matched_delta_unit",
        "dtype": "string",
        "required": True,
        "description": "Unit of matched_delta.",
        "example": "mV",
    },
    {
        "column_name": "window_definition",
        "dtype": "string",
        "required": True,
        "description": "State / voltage / time window used for feature extraction.",
        "example": "central_Q_window_20_80pct",
    },
    {
        "column_name": "preprocessing_setting",
        "dtype": "string",
        "required": True,
        "description": "Smoothing, interpolation, resampling, or derivative preprocessing setting.",
        "example": "savgol_window_31_poly3",
    },
    {
        "column_name": "endpoint_amplification_flag",
        "dtype": "bool",
        "required": True,
        "description": "Whether the observed signal is dominated by endpoint or cutoff-boundary region.",
        "example": "False",
    },
    {
        "column_name": "smoothing_stability_status",
        "dtype": "category",
        "required": True,
        "description": "Stability of feature across smoothing / preprocessing variants.",
        "example": "stable / unstable / not_applicable",
    },
    {
        "column_name": "protocol_sensitivity_status",
        "dtype": "category",
        "required": True,
        "description": "Whether the feature appears protocol-sensitive under the current audit.",
        "example": "unknown / acceptable / sensitive",
    },
    {
        "column_name": "feature_status",
        "dtype": "category",
        "required": True,
        "description": "Feature-level result status before mechanism-level interpretation.",
        "example": "detected / weak / rejected / deferred",
    },
    {
        "column_name": "claim_level",
        "dtype": "category",
        "required": True,
        "description": "Claim ladder level supported by this feature.",
        "example": "Level 1",
    },
    {
        "column_name": "boundary_flag",
        "dtype": "string",
        "required": True,
        "description": "Boundary warning if the feature is endpoint-sensitive, smoothing-sensitive, weak, or execution-limited.",
        "example": "weak_matched_delta",
    },
    {
        "column_name": "notes",
        "dtype": "string",
        "required": False,
        "description": "Free-text notes for audit context.",
        "example": "Central-window signal weak; endpoint region excluded.",
    },
]

diagnostic_schema_df = pd.DataFrame(schema_rows)

diagnostic_schema_data_path = data_dir / "plating_external_diagnostic_schema_v0_1.csv"
diagnostic_schema_docs_path = docs_table_dir / "plating_external_diagnostic_schema_v0_1.csv"

diagnostic_schema_df.to_csv(diagnostic_schema_data_path, index=False)
diagnostic_schema_df.to_csv(diagnostic_schema_docs_path, index=False)

print(f"[OK] saved diagnostic schema: {diagnostic_schema_data_path}")
print(f"[OK] saved docs diagnostic schema: {diagnostic_schema_docs_path}")

print("\n[OK] Diagnostic descriptor schema:")
display(diagnostic_schema_df)

# Diagnostic-layer feature plan
feature_plan_rows = [
    {
        "diagnostic_layer": "C25_VQ",
        "planned_feature_group": "central-window voltage drift",
        "required_features": (
            "central_mean_delta_U_mV; central_median_delta_U_mV; "
            "central_max_abs_delta_U_mV; endpoint_amplification_flag"
        ),
        "primary_risk": "endpoint_amplification",
        "minimum_acceptance_condition": (
            "Matched central-window voltage signal detected and not endpoint-only."
        ),
    },
    {
        "diagnostic_layer": "GITT_like_finite_rest",
        "planned_feature_group": "finite-rest voltage offset",
        "required_features": (
            "finite_rest_delta_U_mV; rest_time_s; state_point_Q_or_SOC; relaxation_drift_flag"
        ),
        "primary_risk": "rest-duration sensitivity",
        "minimum_acceptance_condition": (
            "Matched finite-rest voltage signal persists under defined rest-time window."
        ),
    },
    {
        "diagnostic_layer": "ICA",
        "planned_feature_group": "central incremental-capacity features",
        "required_features": (
            "ICA_peak_shift_central; ICA_area_delta_central; "
            "signal_to_smoothing_ratio; direction_consistency"
        ),
        "primary_risk": "smoothing_sensitivity",
        "minimum_acceptance_condition": (
            "Central ICA feature passes smoothing-stability and matched-control checks."
        ),
    },
    {
        "diagnostic_layer": "DVA",
        "planned_feature_group": "central differential-voltage features",
        "required_features": (
            "DVA_median_central_delta; DVA_peak_shift_central; "
            "DVA_area_delta_central; signal_to_smoothing_ratio; direction_consistency"
        ),
        "primary_risk": "smoothing_sensitivity",
        "minimum_acceptance_condition": (
            "Central DVA feature passes smoothing-stability and matched-control checks."
        ),
    },
]

diagnostic_feature_plan_df = pd.DataFrame(feature_plan_rows)

feature_plan_data_path = data_dir / "plating_external_diagnostic_feature_plan_v0_1.csv"
feature_plan_docs_path = docs_table_dir / "plating_external_diagnostic_feature_plan_v0_1.csv"

diagnostic_feature_plan_df.to_csv(feature_plan_data_path, index=False)
diagnostic_feature_plan_df.to_csv(feature_plan_docs_path, index=False)

print(f"\n[OK] saved diagnostic feature plan: {feature_plan_data_path}")
print(f"[OK] saved docs diagnostic feature plan: {feature_plan_docs_path}")

print("\n[OK] Diagnostic feature plan:")
display(diagnostic_feature_plan_df)

# Audit checks
required_columns = {
    "condition_id",
    "control_condition_id",
    "matched_pair_id",
    "diagnostic_layer",
    "feature_name",
    "feature_value",
    "feature_unit",
    "matched_delta",
    "matched_delta_unit",
    "window_definition",
    "preprocessing_setting",
    "endpoint_amplification_flag",
    "smoothing_stability_status",
    "protocol_sensitivity_status",
    "feature_status",
    "claim_level",
    "boundary_flag",
}

observed_columns = set(diagnostic_schema_df["column_name"])
missing_columns = required_columns - observed_columns

assert not missing_columns, f"Missing required schema columns: {missing_columns}"

expected_layers = {"C25_VQ", "GITT_like_finite_rest", "ICA", "DVA"}
observed_layers = set(diagnostic_feature_plan_df["diagnostic_layer"])

assert observed_layers == expected_layers, (
    f"Diagnostic feature plan layers mismatch: expected {expected_layers}, observed {observed_layers}"
)

assert diagnostic_schema_df["column_name"].is_unique, (
    "Schema column names must be unique."
)

print("\n[BOUNDARY CHECKS]")
print("[OK] Descriptor schema includes matched-control fields.")
print("[OK] Descriptor schema includes window, preprocessing, and boundary fields.")
print("[OK] All four diagnostic layers have feature-plan entries.")
print("[OK] Schema defines reporting requirements only; no feature is supported yet.")

[OK] saved diagnostic schema: /Users/louislu/projects/pybamm-aging-bridge/data/plating_external_diagnostic_schema_v0_1.csv
[OK] saved docs diagnostic schema: /Users/louislu/projects/pybamm-aging-bridge/docs/tables/plating_external_diagnostic_schema_v0_1.csv

[OK] Diagnostic descriptor schema:


,column_name,dtype,required,description,example
0,condition_id,string,True,Condition identifier from the matched-control ...,plating_10C_2C
1,control_condition_id,string,True,Matched no-plating control condition. Empty on...,no_plating_10C_2C
2,matched_pair_id,string,True,Matched pair identifier shared by plating-enab...,pair_10C_2C
3,diagnostic_layer,category,True,Diagnostic layer being evaluated.,C25_VQ
4,feature_name,string,True,Name of extracted diagnostic feature.,central_mean_delta_U
5,feature_value,float,True,Feature value for the current condition.,-2.35
6,feature_unit,string,True,Unit of the feature value.,mV
7,matched_delta,float,True,Plating-enabled feature minus matched no-plati...,-1.12
8,matched_delta_unit,string,True,Unit of matched_delta.,mV
9,window_definition,string,True,State / voltage / time window used for feature...,central_Q_window_20_80pct



[OK] saved diagnostic feature plan: /Users/louislu/projects/pybamm-aging-bridge/data/plating_external_diagnostic_feature_plan_v0_1.csv
[OK] saved docs diagnostic feature plan: /Users/louislu/projects/pybamm-aging-bridge/docs/tables/plating_external_diagnostic_feature_plan_v0_1.csv

[OK] Diagnostic feature plan:


,diagnostic_layer,planned_feature_group,required_features,primary_risk,minimum_acceptance_condition
0,C25_VQ,central-window voltage drift,central_mean_delta_U_mV; central_median_delta_...,endpoint_amplification,Matched central-window voltage signal detected...
1,GITT_like_finite_rest,finite-rest voltage offset,finite_rest_delta_U_mV; rest_time_s; state_poi...,rest-duration sensitivity,Matched finite-rest voltage signal persists un...
2,ICA,central incremental-capacity features,ICA_peak_shift_central; ICA_area_delta_central...,smoothing_sensitivity,Central ICA feature passes smoothing-stability...
3,DVA,central differential-voltage features,DVA_median_central_delta; DVA_peak_shift_centr...,smoothing_sensitivity,Central DVA feature passes smoothing-stability...



[BOUNDARY CHECKS]
[OK] Descriptor schema includes matched-control fields.
[OK] Descriptor schema includes window, preprocessing, and boundary fields.
[OK] All four diagnostic layers have feature-plan entries.
[OK] Schema defines reporting requirements only; no feature is supported yet.


## 4. Decision-rule registry（判据注册表）

This block defines the decision rules for interpreting future lithium-plating external diagnostic outputs.

The decision rules prevent post-hoc claim escalation.

A future diagnostic feature can only support a higher claim level if it passes predefined checks:

- matched-control signal detection（匹配对照信号检出）
- central-window support（中央窗口支持）
- endpoint-amplification exclusion（端点放大排除）
- smoothing-stability check（平滑稳定性检查）
- protocol-boundary check（协议边界检查）
- mechanism-contrast requirement（机制对照要求）

Important boundary:

A feature can be measurable but still rejected as a primary fitting target.

Level 1 and Level 2 claims may be supported by this future workflow. Level 3 fitting-target qualification requires later validation and is not claimed by this contract notebook.

In [5]:
# Cell 10 — Define and save decision-rule registry（定义并保存判据注册表）

decision_rule_rows = [
    {
        "rule_id": "R1_matched_signal_detected",
        "rule_name": "matched-control signal detection",
        "applies_to": "all_diagnostic_layers",
        "input_fields": "matched_delta; matched_delta_unit; feature_status",
        "pass_condition": (
            "matched_delta is quantitatively detectable and feature_status is not rejected or deferred"
        ),
        "fail_condition": (
            "matched_delta is near zero, below protocol sensitivity, or feature_status is weak / rejected / deferred"
        ),
        "if_pass": "eligible_for_Level_1",
        "if_fail": "remain_deferred_or_auxiliary",
        "allowed_claim": (
            "A matched external diagnostic signal is detected under the tested model and protocol conditions."
        ),
        "forbidden_claim": (
            "Do not claim robust fingerprint or fitting-target status from matched delta alone."
        ),
    },
    {
        "rule_id": "R2_central_window_supported",
        "rule_name": "central-window support",
        "applies_to": "C25_VQ; ICA; DVA",
        "input_fields": "window_definition; endpoint_amplification_flag; boundary_flag",
        "pass_condition": (
            "signal is present in the predefined central state window and not dominated by endpoint behavior"
        ),
        "fail_condition": (
            "signal appears only near cutoff, endpoint, or boundary-sensitive region"
        ),
        "if_pass": "eligible_for_Level_2_if_other_checks_pass",
        "if_fail": "endpoint_sensitive_auxiliary_only",
        "allowed_claim": (
            "Central-window diagnostic signal is supported under the tested protocol."
        ),
        "forbidden_claim": (
            "Do not use endpoint-only signal as primary plating fingerprint."
        ),
    },
    {
        "rule_id": "R3_endpoint_amplification_excluded",
        "rule_name": "endpoint-amplification exclusion",
        "applies_to": "C25_VQ; ICA; DVA",
        "input_fields": "endpoint_amplification_flag; boundary_flag",
        "pass_condition": "endpoint_amplification_flag is False",
        "fail_condition": "endpoint_amplification_flag is True",
        "if_pass": "retain_feature_for_Level_2_review",
        "if_fail": "reject_as_primary_target",
        "allowed_claim": (
            "The feature is not dominated by endpoint amplification in the tested window."
        ),
        "forbidden_claim": (
            "Do not treat endpoint-amplified drift as mechanism-specific fitting target."
        ),
    },
    {
        "rule_id": "R4_smoothing_stability",
        "rule_name": "smoothing-stability check",
        "applies_to": "ICA; DVA",
        "input_fields": "preprocessing_setting; smoothing_stability_status",
        "pass_condition": "smoothing_stability_status is stable or not_applicable",
        "fail_condition": "smoothing_stability_status is unstable",
        "if_pass": "retain_derivative_feature_for_Level_2_review",
        "if_fail": "derivative_feature_auxiliary_or_rejected",
        "allowed_claim": (
            "Derivative-layer feature passes smoothing-stability control."
        ),
        "forbidden_claim": (
            "Do not accept ICA / DVA peak changes as primary evidence without smoothing audit."
        ),
    },
    {
        "rule_id": "R5_protocol_boundary",
        "rule_name": "protocol-boundary check",
        "applies_to": "all_diagnostic_layers",
        "input_fields": "protocol_sensitivity_status; boundary_flag; notes",
        "pass_condition": "protocol_sensitivity_status is acceptable or explicitly bounded",
        "fail_condition": "protocol_sensitivity_status is sensitive and not bounded",
        "if_pass": "claim_may_be_protocol_bounded",
        "if_fail": "claim_must_remain_deferred",
        "allowed_claim": (
            "The diagnostic signal is supported within the tested protocol boundary."
        ),
        "forbidden_claim": (
            "Do not generalize to other protocols, chemistries, temperatures, or operating envelopes."
        ),
    },
    {
        "rule_id": "R6_mechanism_contrast_required_for_Level_3",
        "rule_name": "mechanism-contrast requirement for fitting-target candidate",
        "applies_to": "all_diagnostic_layers",
        "input_fields": "claim_level; boundary_flag; notes",
        "pass_condition": (
            "feature is stable and not easily reproduced by simpler SEI / LAM / resistance-growth branches"
        ),
        "fail_condition": (
            "feature is mechanism-non-unique or contrast branches are not available"
        ),
        "if_pass": "eligible_for_Level_3_in_future_validation",
        "if_fail": "maximum_Level_2_in_current_framework",
        "allowed_claim": (
            "The feature may be considered as a fitting-target candidate after mechanism-contrast validation."
        ),
        "forbidden_claim": (
            "Do not claim fitting-target qualification in this contract notebook."
        ),
    },
    {
        "rule_id": "R7_execution_boundary",
        "rule_name": "execution-boundary classification",
        "applies_to": "process_isolated_execution",
        "input_fields": "execution_status; boundary_flag; notes",
        "pass_condition": "execution_status is completed and outputs pass schema checks",
        "fail_condition": "solver failure, kernel death, memory pressure, or incomplete output",
        "if_pass": "outputs_can_enter_descriptor_audit",
        "if_fail": "classify_as_execution_boundary_not_scientific_negative",
        "allowed_claim": (
            "Execution failed or was incomplete under the tested workflow."
        ),
        "forbidden_claim": (
            "Do not interpret solver failure or kernel death as absence of plating diagnostic signal."
        ),
    },
]

decision_rules_df = pd.DataFrame(decision_rule_rows)

decision_rules_data_path = data_dir / "plating_external_diagnostic_decision_rules_v0_1.csv"
decision_rules_docs_path = docs_table_dir / "plating_external_diagnostic_decision_rules_v0_1.csv"

decision_rules_df.to_csv(decision_rules_data_path, index=False)
decision_rules_df.to_csv(decision_rules_docs_path, index=False)

print(f"[OK] saved decision-rule registry: {decision_rules_data_path}")
print(f"[OK] saved docs decision-rule registry: {decision_rules_docs_path}")

print("\n[OK] Decision-rule registry:")
display(decision_rules_df)

# Compact audit table
compact_decision_rules_df = decision_rules_df[
    [
        "rule_id",
        "rule_name",
        "applies_to",
        "if_pass",
        "if_fail",
        "forbidden_claim",
    ]
].copy()

print("\n[OK] Compact decision-rule registry:")
display(compact_decision_rules_df)

# Audit checks
required_rules = {
    "R1_matched_signal_detected",
    "R2_central_window_supported",
    "R3_endpoint_amplification_excluded",
    "R4_smoothing_stability",
    "R5_protocol_boundary",
    "R6_mechanism_contrast_required_for_Level_3",
    "R7_execution_boundary",
}

observed_rules = set(decision_rules_df["rule_id"])
missing_rules = required_rules - observed_rules

assert not missing_rules, f"Missing decision rules: {missing_rules}"

assert decision_rules_df["rule_id"].is_unique, "Decision rule IDs must be unique."

assert decision_rules_df["forbidden_claim"].notna().all(), (
    "Every decision rule must include a forbidden claim."
)

level3_rule = decision_rules_df[
    decision_rules_df["rule_id"] == "R6_mechanism_contrast_required_for_Level_3"
].iloc[0]

assert "Do not claim fitting-target qualification" in level3_rule["forbidden_claim"], (
    "Level 3 rule must explicitly forbid fitting-target qualification in this contract."
)

execution_rule = decision_rules_df[
    decision_rules_df["rule_id"] == "R7_execution_boundary"
].iloc[0]

assert "absence of plating diagnostic signal" in execution_rule["forbidden_claim"], (
    "Execution-boundary rule must prevent scientific negative overclaim."
)

print("\n[BOUNDARY CHECKS]")
print("[OK] Required decision rules are present.")
print("[OK] Every rule defines allowed and forbidden claim wording.")
print("[OK] Level 3 fitting-target qualification is explicitly blocked in this contract.")
print("[OK] Execution failure is classified as execution boundary, not scientific negative evidence.")
print("[OK] Decision rules prevent post-hoc claim escalation.")

[OK] saved decision-rule registry: /Users/louislu/projects/pybamm-aging-bridge/data/plating_external_diagnostic_decision_rules_v0_1.csv
[OK] saved docs decision-rule registry: /Users/louislu/projects/pybamm-aging-bridge/docs/tables/plating_external_diagnostic_decision_rules_v0_1.csv

[OK] Decision-rule registry:


,rule_id,rule_name,applies_to,input_fields,pass_condition,fail_condition,if_pass,if_fail,allowed_claim,forbidden_claim
0,R1_matched_signal_detected,matched-control signal detection,all_diagnostic_layers,matched_delta; matched_delta_unit; feature_status,matched_delta is quantitatively detectable and...,"matched_delta is near zero, below protocol sen...",eligible_for_Level_1,remain_deferred_or_auxiliary,A matched external diagnostic signal is detect...,Do not claim robust fingerprint or fitting-tar...
1,R2_central_window_supported,central-window support,C25_VQ; ICA; DVA,window_definition; endpoint_amplification_flag...,signal is present in the predefined central st...,"signal appears only near cutoff, endpoint, or ...",eligible_for_Level_2_if_other_checks_pass,endpoint_sensitive_auxiliary_only,Central-window diagnostic signal is supported ...,Do not use endpoint-only signal as primary pla...
2,R3_endpoint_amplification_excluded,endpoint-amplification exclusion,C25_VQ; ICA; DVA,endpoint_amplification_flag; boundary_flag,endpoint_amplification_flag is False,endpoint_amplification_flag is True,retain_feature_for_Level_2_review,reject_as_primary_target,The feature is not dominated by endpoint ampli...,Do not treat endpoint-amplified drift as mecha...
3,R4_smoothing_stability,smoothing-stability check,ICA; DVA,preprocessing_setting; smoothing_stability_status,smoothing_stability_status is stable or not_ap...,smoothing_stability_status is unstable,retain_derivative_feature_for_Level_2_review,derivative_feature_auxiliary_or_rejected,Derivative-layer feature passes smoothing-stab...,Do not accept ICA / DVA peak changes as primar...
4,R5_protocol_boundary,protocol-boundary check,all_diagnostic_layers,protocol_sensitivity_status; boundary_flag; notes,protocol_sensitivity_status is acceptable or e...,protocol_sensitivity_status is sensitive and n...,claim_may_be_protocol_bounded,claim_must_remain_deferred,The diagnostic signal is supported within the ...,"Do not generalize to other protocols, chemistr..."
5,R6_mechanism_contrast_required_for_Level_3,mechanism-contrast requirement for fitting-tar...,all_diagnostic_layers,claim_level; boundary_flag; notes,feature is stable and not easily reproduced by...,feature is mechanism-non-unique or contrast br...,eligible_for_Level_3_in_future_validation,maximum_Level_2_in_current_framework,The feature may be considered as a fitting-tar...,Do not claim fitting-target qualification in t...
6,R7_execution_boundary,execution-boundary classification,process_isolated_execution,execution_status; boundary_flag; notes,execution_status is completed and outputs pass...,"solver failure, kernel death, memory pressure,...",outputs_can_enter_descriptor_audit,classify_as_execution_boundary_not_scientific_...,Execution failed or was incomplete under the t...,Do not interpret solver failure or kernel deat...



[OK] Compact decision-rule registry:


,rule_id,rule_name,applies_to,if_pass,if_fail,forbidden_claim
0,R1_matched_signal_detected,matched-control signal detection,all_diagnostic_layers,eligible_for_Level_1,remain_deferred_or_auxiliary,Do not claim robust fingerprint or fitting-tar...
1,R2_central_window_supported,central-window support,C25_VQ; ICA; DVA,eligible_for_Level_2_if_other_checks_pass,endpoint_sensitive_auxiliary_only,Do not use endpoint-only signal as primary pla...
2,R3_endpoint_amplification_excluded,endpoint-amplification exclusion,C25_VQ; ICA; DVA,retain_feature_for_Level_2_review,reject_as_primary_target,Do not treat endpoint-amplified drift as mecha...
3,R4_smoothing_stability,smoothing-stability check,ICA; DVA,retain_derivative_feature_for_Level_2_review,derivative_feature_auxiliary_or_rejected,Do not accept ICA / DVA peak changes as primar...
4,R5_protocol_boundary,protocol-boundary check,all_diagnostic_layers,claim_may_be_protocol_bounded,claim_must_remain_deferred,"Do not generalize to other protocols, chemistr..."
5,R6_mechanism_contrast_required_for_Level_3,mechanism-contrast requirement for fitting-tar...,all_diagnostic_layers,eligible_for_Level_3_in_future_validation,maximum_Level_2_in_current_framework,Do not claim fitting-target qualification in t...
6,R7_execution_boundary,execution-boundary classification,process_isolated_execution,outputs_can_enter_descriptor_audit,classify_as_execution_boundary_not_scientific_...,Do not interpret solver failure or kernel deat...



[BOUNDARY CHECKS]
[OK] Required decision rules are present.
[OK] Every rule defines allowed and forbidden claim wording.
[OK] Level 3 fitting-target qualification is explicitly blocked in this contract.
[OK] Execution failure is classified as execution boundary, not scientific negative evidence.
[OK] Decision rules prevent post-hoc claim escalation.


## 5. Process-isolated execution plan（进程隔离执行计划）

This block defines the execution plan for future lithium-plating external diagnostic simulations.

Previous plating workflows showed that dense in-notebook execution can exceed stable notebook limits when combining:

- DFN model
- SEI background
- lithium plating
- `starting_solution`
- dense HPPC-like or derivative diagnostic sampling

Therefore, future diagnostic simulations must be executed outside the notebook.

Execution principle:

- one condition per external Python process
- one diagnostic layer per output bundle
- CSV outputs only
- notebook reads completed CSV files for audit
- failed runs are classified as execution boundaries, not scientific negative results

This notebook does not call the process-isolated scripts. It only defines the planned execution structure.

In [6]:
# Cell 12 — Define and save process-isolated execution plan（定义并保存进程隔离执行计划）

execution_plan_rows = [
    {
        "execution_stage": "stage_0_contract",
        "script_name": "not_applicable",
        "execution_location": "notebook_contract_only",
        "input_source": "condition_registry; schema; decision_rules",
        "output_expected": (
            "plating_external_claim_ladder_v0_1.csv; "
            "plating_external_diagnostic_condition_registry_v0_1.csv; "
            "plating_external_diagnostic_schema_v0_1.csv; "
            "plating_external_diagnostic_decision_rules_v0_1.csv"
        ),
        "memory_risk": "none",
        "execution_status": "completed_by_contract_notebook",
        "failure_classification": "not_applicable",
        "notes": "Current notebook freezes the contract only and does not run simulations.",
    },
    {
        "execution_stage": "stage_1_stress_state_generation",
        "script_name": "scripts/run_plating_external_stress_condition.py",
        "execution_location": "external_python_process",
        "input_source": "plating_external_diagnostic_condition_registry_v0_1.csv",
        "output_expected": (
            "per-condition post-stress state summary CSV; direct plating-state audit CSV; "
            "minimal serialized state reference if needed"
        ),
        "memory_risk": "high",
        "execution_status": "planned",
        "failure_classification": "execution_boundary_not_scientific_negative",
        "notes": (
            "Generate post-stress states one condition at a time. Avoid keeping large Solution objects in notebook memory."
        ),
    },
    {
        "execution_stage": "stage_2_C25_VQ_diagnostic",
        "script_name": "scripts/run_plating_external_c25_vq_condition.py",
        "execution_location": "external_python_process",
        "input_source": "post-stress state reference per condition",
        "output_expected": (
            "C/25 V(Q) curve CSV; central-window descriptor CSV; endpoint-amplification audit CSV"
        ),
        "memory_risk": "moderate_to_high",
        "execution_status": "planned",
        "failure_classification": "execution_boundary_not_scientific_negative",
        "notes": "Run one condition per process; export curves and descriptors only.",
    },
    {
        "execution_stage": "stage_3_GITT_like_finite_rest_diagnostic",
        "script_name": "scripts/run_plating_external_gitt_like_condition.py",
        "execution_location": "external_python_process",
        "input_source": "post-stress state reference per condition",
        "output_expected": (
            "finite-rest voltage point CSV; rest-time descriptor CSV; relaxation-drift audit CSV"
        ),
        "memory_risk": "moderate_to_high",
        "execution_status": "planned",
        "failure_classification": "execution_boundary_not_scientific_negative",
        "notes": "Rest-time sensitivity must be explicit; finite-rest voltage must not be overinterpreted as equilibrium OCV.",
    },
    {
        "execution_stage": "stage_4_ICA_DVA_descriptor_extraction",
        "script_name": "scripts/extract_plating_external_ica_dva_descriptors.py",
        "execution_location": "external_python_process_or_lightweight_notebook_after_csv",
        "input_source": "C/25 V(Q) curve CSV files",
        "output_expected": (
            "ICA descriptor CSV; DVA descriptor CSV; smoothing-sensitivity audit CSV"
        ),
        "memory_risk": "low_after_curve_export",
        "execution_status": "planned",
        "failure_classification": "execution_boundary_not_scientific_negative",
        "notes": (
            "Derivative extraction should operate on exported curves, not live PyBaMM Solution objects."
        ),
    },
    {
        "execution_stage": "stage_5_descriptor_merge_and_decision",
        "script_name": "notebooks/29_or_later_descriptor_merge_and_decision_audit.ipynb",
        "execution_location": "notebook_csv_audit_only",
        "input_source": (
            "descriptor CSV files from C/25, GITT-like, ICA, and DVA scripts"
        ),
        "output_expected": (
            "plating_external_diagnostic_descriptors_v0_1.csv; "
            "plating_external_diagnostic_decision_table_v0_1.csv"
        ),
        "memory_risk": "low",
        "execution_status": "planned",
        "failure_classification": "not_applicable",
        "notes": (
            "Notebook should only merge CSV outputs, apply decision rules, and produce figures/tables."
        ),
    },
]

execution_plan_df = pd.DataFrame(execution_plan_rows)

execution_plan_data_path = data_dir / "plating_external_process_isolated_execution_plan_v0_1.csv"
execution_plan_docs_path = docs_table_dir / "plating_external_process_isolated_execution_plan_v0_1.csv"

execution_plan_df.to_csv(execution_plan_data_path, index=False)
execution_plan_df.to_csv(execution_plan_docs_path, index=False)

print(f"[OK] saved execution plan: {execution_plan_data_path}")
print(f"[OK] saved docs execution plan: {execution_plan_docs_path}")

print("\n[OK] Process-isolated execution plan:")
display(execution_plan_df)

# Compact execution roadmap
compact_execution_plan_df = execution_plan_df[
    [
        "execution_stage",
        "script_name",
        "execution_location",
        "memory_risk",
        "execution_status",
        "failure_classification",
    ]
].copy()

print("\n[OK] Compact execution roadmap:")
display(compact_execution_plan_df)

# Audit checks
assert execution_plan_df["execution_stage"].is_unique, (
    "Each execution stage must be unique."
)

planned_external = execution_plan_df[
    execution_plan_df["execution_location"].str.contains("external_python_process", regex=False)
]

assert len(planned_external) >= 3, (
    "At least three future external-process stages should be planned."
)

assert (
    planned_external["failure_classification"]
    .eq("execution_boundary_not_scientific_negative")
    .all()
), "All external-process failures must be classified as execution boundaries."

assert (
    execution_plan_df.loc[
        execution_plan_df["execution_stage"] == "stage_5_descriptor_merge_and_decision",
        "memory_risk"
    ]
    .iloc[0]
    == "low"
), "Final descriptor merge should be low-memory CSV audit only."

assert (
    execution_plan_df["notes"]
    .str.contains("Solution objects", case=False, regex=False)
    .any()
), "Execution plan must explicitly avoid live large Solution objects."

print("\n[BOUNDARY CHECKS]")
print("[OK] Future heavy diagnostics are assigned to external Python processes.")
print("[OK] Notebook-level future work is restricted to CSV audit / decision logic.")
print("[OK] Execution failures are classified as execution boundaries, not scientific negatives.")
print("[OK] Derivative extraction is planned after curve export, not on live Solution objects.")

[OK] saved execution plan: /Users/louislu/projects/pybamm-aging-bridge/data/plating_external_process_isolated_execution_plan_v0_1.csv
[OK] saved docs execution plan: /Users/louislu/projects/pybamm-aging-bridge/docs/tables/plating_external_process_isolated_execution_plan_v0_1.csv

[OK] Process-isolated execution plan:


,execution_stage,script_name,execution_location,input_source,output_expected,memory_risk,execution_status,failure_classification,notes
0,stage_0_contract,not_applicable,notebook_contract_only,condition_registry; schema; decision_rules,plating_external_claim_ladder_v0_1.csv; platin...,none,completed_by_contract_notebook,not_applicable,Current notebook freezes the contract only and...
1,stage_1_stress_state_generation,scripts/run_plating_external_stress_condition.py,external_python_process,plating_external_diagnostic_condition_registry...,per-condition post-stress state summary CSV; d...,high,planned,execution_boundary_not_scientific_negative,Generate post-stress states one condition at a...
2,stage_2_C25_VQ_diagnostic,scripts/run_plating_external_c25_vq_condition.py,external_python_process,post-stress state reference per condition,C/25 V(Q) curve CSV; central-window descriptor...,moderate_to_high,planned,execution_boundary_not_scientific_negative,Run one condition per process; export curves a...
3,stage_3_GITT_like_finite_rest_diagnostic,scripts/run_plating_external_gitt_like_conditi...,external_python_process,post-stress state reference per condition,finite-rest voltage point CSV; rest-time descr...,moderate_to_high,planned,execution_boundary_not_scientific_negative,Rest-time sensitivity must be explicit; finite...
4,stage_4_ICA_DVA_descriptor_extraction,scripts/extract_plating_external_ica_dva_descr...,external_python_process_or_lightweight_noteboo...,C/25 V(Q) curve CSV files,ICA descriptor CSV; DVA descriptor CSV; smooth...,low_after_curve_export,planned,execution_boundary_not_scientific_negative,Derivative extraction should operate on export...
5,stage_5_descriptor_merge_and_decision,notebooks/29_or_later_descriptor_merge_and_dec...,notebook_csv_audit_only,"descriptor CSV files from C/25, GITT-like, ICA...",plating_external_diagnostic_descriptors_v0_1.c...,low,planned,not_applicable,"Notebook should only merge CSV outputs, apply ..."



[OK] Compact execution roadmap:


,execution_stage,script_name,execution_location,memory_risk,execution_status,failure_classification
0,stage_0_contract,not_applicable,notebook_contract_only,none,completed_by_contract_notebook,not_applicable
1,stage_1_stress_state_generation,scripts/run_plating_external_stress_condition.py,external_python_process,high,planned,execution_boundary_not_scientific_negative
2,stage_2_C25_VQ_diagnostic,scripts/run_plating_external_c25_vq_condition.py,external_python_process,moderate_to_high,planned,execution_boundary_not_scientific_negative
3,stage_3_GITT_like_finite_rest_diagnostic,scripts/run_plating_external_gitt_like_conditi...,external_python_process,moderate_to_high,planned,execution_boundary_not_scientific_negative
4,stage_4_ICA_DVA_descriptor_extraction,scripts/extract_plating_external_ica_dva_descr...,external_python_process_or_lightweight_noteboo...,low_after_curve_export,planned,execution_boundary_not_scientific_negative
5,stage_5_descriptor_merge_and_decision,notebooks/29_or_later_descriptor_merge_and_dec...,notebook_csv_audit_only,low,planned,not_applicable



[BOUNDARY CHECKS]
[OK] Future heavy diagnostics are assigned to external Python processes.
[OK] Notebook-level future work is restricted to CSV audit / decision logic.
[OK] Execution failures are classified as execution boundaries, not scientific negatives.
[OK] Derivative extraction is planned after curve export, not on live Solution objects.


In [7]:
# Cell 13 — Build and save contract output inventory（构建并保存契约输出清单）

contract_outputs = [
    {
        "artifact_name": "claim_ladder",
        "artifact_role": "Defines evidence levels from model-internal observability to fitting-target candidate.",
        "data_path": "data/plating_external_claim_ladder_v0_1.csv",
        "docs_path": "docs/tables/plating_external_claim_ladder_v0_1.csv",
        "produced_by_cell": "Cell 4",
        "artifact_status": "contract_output",
    },
    {
        "artifact_name": "condition_registry",
        "artifact_role": "Defines matched plating / no-plating condition pairs.",
        "data_path": "data/plating_external_diagnostic_condition_registry_v0_1.csv",
        "docs_path": "docs/tables/plating_external_diagnostic_condition_registry_v0_1.csv",
        "produced_by_cell": "Cell 6",
        "artifact_status": "contract_output",
    },
    {
        "artifact_name": "diagnostic_schema",
        "artifact_role": "Defines required descriptor table columns for future diagnostic outputs.",
        "data_path": "data/plating_external_diagnostic_schema_v0_1.csv",
        "docs_path": "docs/tables/plating_external_diagnostic_schema_v0_1.csv",
        "produced_by_cell": "Cell 8",
        "artifact_status": "contract_output",
    },
    {
        "artifact_name": "diagnostic_feature_plan",
        "artifact_role": "Defines planned feature groups for C/25 V(Q), GITT-like voltage, ICA, and DVA.",
        "data_path": "data/plating_external_diagnostic_feature_plan_v0_1.csv",
        "docs_path": "docs/tables/plating_external_diagnostic_feature_plan_v0_1.csv",
        "produced_by_cell": "Cell 8",
        "artifact_status": "contract_output",
    },
    {
        "artifact_name": "decision_rules",
        "artifact_role": "Defines predefined claim-escalation and rejection rules.",
        "data_path": "data/plating_external_diagnostic_decision_rules_v0_1.csv",
        "docs_path": "docs/tables/plating_external_diagnostic_decision_rules_v0_1.csv",
        "produced_by_cell": "Cell 10",
        "artifact_status": "contract_output",
    },
    {
        "artifact_name": "process_isolated_execution_plan",
        "artifact_role": "Defines future external-process execution stages and execution-boundary handling.",
        "data_path": "data/plating_external_process_isolated_execution_plan_v0_1.csv",
        "docs_path": "docs/tables/plating_external_process_isolated_execution_plan_v0_1.csv",
        "produced_by_cell": "Cell 12",
        "artifact_status": "contract_output",
    },
]

contract_inventory_df = pd.DataFrame(contract_outputs)

# Resolve paths and audit existence
contract_inventory_df["data_exists"] = contract_inventory_df["data_path"].map(lambda p: (repo / p).exists())
contract_inventory_df["docs_exists"] = contract_inventory_df["docs_path"].map(lambda p: (repo / p).exists())

# Add row-count audit for existing CSV files
def safe_row_count(relative_path):
    path = repo / relative_path
    if not path.exists():
        return np.nan
    try:
        return len(pd.read_csv(path))
    except Exception:
        return np.nan

contract_inventory_df["data_n_rows"] = contract_inventory_df["data_path"].map(safe_row_count)
contract_inventory_df["docs_n_rows"] = contract_inventory_df["docs_path"].map(safe_row_count)

inventory_data_path = data_dir / "plating_external_contract_inventory_v0_1.csv"
inventory_docs_path = docs_table_dir / "plating_external_contract_inventory_v0_1.csv"

contract_inventory_df.to_csv(inventory_data_path, index=False)
contract_inventory_df.to_csv(inventory_docs_path, index=False)

print(f"[OK] saved contract inventory: {inventory_data_path}")
print(f"[OK] saved docs contract inventory: {inventory_docs_path}")

print("\n[OK] Contract output inventory:")
display(contract_inventory_df)

# Compact audit table
compact_inventory_df = contract_inventory_df[
    [
        "artifact_name",
        "produced_by_cell",
        "data_exists",
        "docs_exists",
        "data_n_rows",
        "docs_n_rows",
    ]
].copy()

print("\n[OK] Compact contract output audit:")
display(compact_inventory_df)

# Audit checks
assert contract_inventory_df["data_exists"].all(), (
    "All data/ contract outputs must exist."
)

assert contract_inventory_df["docs_exists"].all(), (
    "All docs/tables/ contract outputs must exist."
)

assert contract_inventory_df["data_n_rows"].notna().all(), (
    "All data/ contract outputs must be readable CSV files."
)

assert contract_inventory_df["docs_n_rows"].notna().all(), (
    "All docs/tables/ contract outputs must be readable CSV files."
)

assert (
    contract_inventory_df["data_n_rows"].equals(contract_inventory_df["docs_n_rows"])
), "data/ and docs/tables/ versions must have identical row counts."

# Contract should not contain simulation-result descriptor outputs yet.
forbidden_result_files = [
    data_dir / "plating_external_diagnostic_descriptors_v0_1.csv",
    data_dir / "plating_external_diagnostic_decision_table_v0_1.csv",
]

unexpected_existing = [p for p in forbidden_result_files if p.exists()]
assert not unexpected_existing, (
    "Result-level files should not exist in this contract notebook: "
    + ", ".join(str(p) for p in unexpected_existing)
)

print("\n[BOUNDARY CHECKS]")
print("[OK] All contract-level artifacts exist in data/ and docs/tables/.")
print("[OK] All contract CSV files are readable.")
print("[OK] data/ and docs/tables/ versions have matching row counts.")
print("[OK] No result-level diagnostic descriptor or decision table is generated in this notebook.")
print("[OK] Notebook 28 remains a contract notebook, not a simulation-result notebook.")

[OK] saved contract inventory: /Users/louislu/projects/pybamm-aging-bridge/data/plating_external_contract_inventory_v0_1.csv
[OK] saved docs contract inventory: /Users/louislu/projects/pybamm-aging-bridge/docs/tables/plating_external_contract_inventory_v0_1.csv

[OK] Contract output inventory:


,artifact_name,artifact_role,data_path,docs_path,produced_by_cell,artifact_status,data_exists,docs_exists,data_n_rows,docs_n_rows
0,claim_ladder,Defines evidence levels from model-internal ob...,data/plating_external_claim_ladder_v0_1.csv,docs/tables/plating_external_claim_ladder_v0_1...,Cell 4,contract_output,True,True,4,4
1,condition_registry,Defines matched plating / no-plating condition...,data/plating_external_diagnostic_condition_reg...,docs/tables/plating_external_diagnostic_condit...,Cell 6,contract_output,True,True,6,6
2,diagnostic_schema,Defines required descriptor table columns for ...,data/plating_external_diagnostic_schema_v0_1.csv,docs/tables/plating_external_diagnostic_schema...,Cell 8,contract_output,True,True,18,18
3,diagnostic_feature_plan,"Defines planned feature groups for C/25 V(Q), ...",data/plating_external_diagnostic_feature_plan_...,docs/tables/plating_external_diagnostic_featur...,Cell 8,contract_output,True,True,4,4
4,decision_rules,Defines predefined claim-escalation and reject...,data/plating_external_diagnostic_decision_rule...,docs/tables/plating_external_diagnostic_decisi...,Cell 10,contract_output,True,True,7,7
5,process_isolated_execution_plan,Defines future external-process execution stag...,data/plating_external_process_isolated_executi...,docs/tables/plating_external_process_isolated_...,Cell 12,contract_output,True,True,6,6



[OK] Compact contract output audit:


,artifact_name,produced_by_cell,data_exists,docs_exists,data_n_rows,docs_n_rows
0,claim_ladder,Cell 4,True,True,4,4
1,condition_registry,Cell 6,True,True,6,6
2,diagnostic_schema,Cell 8,True,True,18,18
3,diagnostic_feature_plan,Cell 8,True,True,4,4
4,decision_rules,Cell 10,True,True,7,7
5,process_isolated_execution_plan,Cell 12,True,True,6,6



[BOUNDARY CHECKS]
[OK] All contract-level artifacts exist in data/ and docs/tables/.
[OK] All contract CSV files are readable.
[OK] data/ and docs/tables/ versions have matching row counts.
[OK] No result-level diagnostic descriptor or decision table is generated in this notebook.
[OK] Notebook 28 remains a contract notebook, not a simulation-result notebook.


## Final closure（最终闭合）

### Key findings

This notebook defined the design contract for lithium-plating external diagnostic closure.

No new PyBaMM simulations were executed.

The notebook established:

- claim ladder from model-internal plating-state observability to fitting-target candidate
- matched-control condition registry
- diagnostic descriptor schema
- diagnostic feature plan for C/25 V(Q), GITT-like finite-rest voltage, ICA, and DVA
- predefined decision-rule registry
- process-isolated execution plan
- contract output inventory

The generated contract artifacts are:

- `data/plating_external_claim_ladder_v0_1.csv`
- `data/plating_external_diagnostic_condition_registry_v0_1.csv`
- `data/plating_external_diagnostic_schema_v0_1.csv`
- `data/plating_external_diagnostic_feature_plan_v0_1.csv`
- `data/plating_external_diagnostic_decision_rules_v0_1.csv`
- `data/plating_external_process_isolated_execution_plan_v0_1.csv`
- `data/plating_external_contract_inventory_v0_1.csv`
- `docs/tables/plating_external_claim_ladder_v0_1.csv`
- `docs/tables/plating_external_diagnostic_condition_registry_v0_1.csv`
- `docs/tables/plating_external_diagnostic_schema_v0_1.csv`
- `docs/tables/plating_external_diagnostic_feature_plan_v0_1.csv`
- `docs/tables/plating_external_diagnostic_decision_rules_v0_1.csv`
- `docs/tables/plating_external_process_isolated_execution_plan_v0_1.csv`
- `docs/tables/plating_external_contract_inventory_v0_1.csv`

### Supported conclusions

Model-internal lithium-plating state observability remains supported from previous plating audits.

The future diagnostic closure workflow must use matched no-plating controls.

Future heavy plating diagnostics must use process-isolated execution.

The external diagnostic layers to be audited are:

1. C/25 central-window V(Q)
2. GITT-like finite-rest voltage
3. ICA central features
4. DVA central features

### Partially supported conclusions

The condition matrix is ready for future execution because each plating-enabled condition has a matched no-plating control at the same temperature and stress charge rate.

The diagnostic schema is ready for future descriptor outputs because it includes matched-control fields, window definitions, preprocessing settings, endpoint-amplification flags, smoothing-stability status, protocol-boundary status, claim level, and boundary flags.

### Unsupported / deferred claims

External lithium-plating diagnostic fingerprint closure remains deferred.

No measurement-accessible lithium-plating fitting target is supported by this notebook.

Level 3 fitting-target qualification is not claimed.

ICA / DVA features are not accepted as primary targets unless future smoothing-stability and matched-control checks pass.

Endpoint-only voltage or derivative signals are not accepted as primary fitting targets.

### Execution or protocol boundaries

This notebook executed no simulation and generated no diagnostic-result descriptors.

Future DFN + SEI-background + lithium-plating workflows may exceed stable notebook execution limits if run directly in Jupyter.

Solver failure, kernel death, memory pressure, or incomplete outputs must be classified as execution boundaries, not as scientific negative evidence.

Future diagnostic scripts must export CSV files and avoid retaining large live PyBaMM Solution objects inside notebooks.

### Correct conclusion wording

The correct conclusion of this notebook is:

A design contract for lithium-plating external diagnostic closure has been established. Model-internal plating-state observability remains supported from previous audits, while measurement-accessible external diagnostic fingerprint closure remains deferred until future process-isolated diagnostic execution produces matched-control descriptor outputs.

### Next step

The next technical step is to implement the first process-isolated execution script:

`scripts/run_plating_external_stress_condition.py`

This script should generate post-stress state summaries and direct plating-state audit summaries for each registered condition, one condition per external Python process.

Only after stage 1 is stable should the C/25 V(Q), GITT-like finite-rest voltage, ICA, and DVA diagnostic layers be executed.